### Javier Carballal Morgade
### Claudia Reyero Bardelás

Simon’s Algorithm.
- Implement Simon’s algorithm from scratch using Qiskit.
- Test it with multiple different hidden strings.
- Comment your code extensively and describe in textual markdown cells the results obtained.

In [5]:
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from itertools import product

First we create a circuit that implements the Simon's algorithm

In [23]:
def simons(f,n):
    # Create circuit with n input registers and n output registers initialized to 0
    circuit = QuantumCircuit(n + n, n)
    circuit.barrier()
    # Apply H to intput qubits
    for i in range(n):
        circuit.h(i)
    circuit.barrier()
    # Apply oracle that transforms: |y>|0> -> |y>|f(x)>
    for x in product([0, 1], repeat=n):
        y = f(x)
        for j in range(n):
            if y[j] == 1:
                for i in range(n):
                    if x[i] == 0:
                        circuit.x(i)
                if n == 1:
                    circuit.cx(0, n + j)
                else:
                    circuit.mcx(list(range(n)), n + j)
                for i in range(n):
                    if x[i] == 0:
                        circuit.x(i)
    circuit.barrier()
    # Apply H to input qubits
    for i in range(n):
        circuit.h(i)
    # Measure input qubits
    for i in range(n):
        circuit.measure(i, i)
    return circuit

Then we create a function that takes the secret and creates a function that hides de secret

In [26]:
def create_f(s):
    p = next((i for i, b in enumerate(s) if b == 1), None)

    def f(x):
        if p is None or x[p] == 0:
            return tuple(x)
        # x XOR s
        return tuple(xi ^ si for xi, si in zip(x, s))
    return f

Then, we create a function that solves the secret. This is the classical post-processing.

In [27]:
def classical_proccess(counts, n):
    # Find s so that y·s = 0 (mod 2) for all of the y measured
    ys = [tuple(int(b) for b in reversed(k)) for k in counts]
    for s in product([0, 1], repeat=n):
        if any(s) and all(sum(a * b for a, b in zip(y, s)) % 2 == 0 for y in ys):
            return s
    return (0,) * n  

Now we test the algorithm for different hidden strings

In [28]:
secrets = [
    (1, 1),
    (1, 0, 1),
    (1, 1, 0),
    (1, 0, 1, 1)    
]

We execute all of the examples for the different hidden strings. The result must be that the algorithms finds the hidden strings within the functions

In [34]:
for s in secrets:
    n = len(s)
    # Create function
    f = create_f(s)
    # Apply algorithm
    qc = simons(f, n)
    # Print circuit
    print(f"secret: {s}")
    print(simons(f, n).draw())
    # Call simulator
    sim = AerSimulator()
    # Run simulator
    counts = sim.run(transpile(qc, sim), shots=1024).result().get_counts()
    # Apply classical post-proccessing
    result = classical_proccess(counts, n)
    print(f"secret: {s} | hiddent string found: {result}")

secret: (1, 1)
      ░ ┌───┐ ░ ┌───┐     ┌───┐           ░ ┌───┐┌─┐   
q_0: ─░─┤ H ├─░─┤ X ├──■──┤ X ├──■────────░─┤ H ├┤M├───
      ░ ├───┤ ░ └───┘  │  ├───┤  │  ┌───┐ ░ ├───┤└╥┘┌─┐
q_1: ─░─┤ H ├─░────────■──┤ X ├──■──┤ X ├─░─┤ H ├─╫─┤M├
      ░ └───┘ ░        │  └───┘  │  └───┘ ░ └───┘ ║ └╥┘
q_2: ─░───────░────────┼─────────┼────────░───────╫──╫─
      ░       ░      ┌─┴─┐     ┌─┴─┐      ░       ║  ║ 
q_3: ─░───────░──────┤ X ├─────┤ X ├──────░───────╫──╫─
      ░       ░      └───┘     └───┘      ░       ║  ║ 
c: 2/═════════════════════════════════════════════╩══╩═
                                                  0  1 
secret: (1, 1) | hiddent string found: (1, 1)
secret: (1, 0, 1)
      ░ ┌───┐ ░ ┌───┐     ┌───┐┌───┐     ┌───┐┌───┐     ┌───┐┌───┐     ┌───┐»
q_0: ─░─┤ H ├─░─┤ X ├──■──┤ X ├┤ X ├──■──┤ X ├┤ X ├──■──┤ X ├┤ X ├──■──┤ X ├»
      ░ ├───┤ ░ ├───┤  │  ├───┤└───┘  │  └───┘└───┘  │  └───┘└───┘  │  ├───┤»
q_1: ─░─┤ H ├─░─┤ X ├──■──┤ X ├───────■──────────────■──────────────■──

We can see that for every hidden string, the Simon algorithm has been able to determine the secret string $s$ correctly